# 输入驱动的物理热记忆：无电子反馈 optical RC

本框架把温度作为材料内部动态状态：外部输入驱动热状态演化，同一输入通过 SLM 编码弱 probe，热状态通过局部相位屏改变固定的散射传播，CCD speckle 给出可观测特征。

**没有 CCD→SLM 电子反馈，没有 state/input 拼接。** SLM 整个有效区域仅编码当前输入；只训练数字线性 readout。需要同目录的 `optical_physical_memory.py`、`MackeyGlass_t17.txt`，及 NumPy、PyTorch、Matplotlib、Jupyter。

## 1. 状态方程和因果顺序

\[
a=e^{-\Delta t/\tau},\qquad
\theta_{k+1}=a\theta_k+\beta u_k,
\qquad r_k=H(\theta_k,u_k),\qquad \hat y_k=W_{\rm out}[1;r_k].
\]

**每个时间步**：① 用当前输入 \(u_k\) 和曝光前温度 \(\theta_k\) 产生 speckle 特征 \(r_k\)；② 当前加热写入 \(\theta_{k+1}\)。因而 \(r_k\) 可包含当前输入和 \(u_0,\ldots,u_{k-1}\) 的热痕迹；本 notebook 的一步预测目标为 \(y_k=u_{k+1}\)。

实现中可直接指定 `beta`。默认 `beta=None` 时取 \(\beta=(1-a)g_{\rm th}\)：这仍是线性 \(\beta u_k\)，且改变 \(\tau\) 时常量输入下的稳态增益 \(g_{\rm th}\) 保持一致。如果实际实验要求固定每次注入量，应显式给 `beta`。

\(u_k\) 必须控制非负吸收功率；代码假设同一归一化输入驱动 heater，同时编码 probe。两束光或输入相关吸收如何耦合是**建模假设**，需要在装置中实现与标定；相位 SLM 本身不会自动保证这种耦合。

## 2. 光学映射与观测

\[
E_{\rm SLM,k}=e^{i\pi\Phi(u_k)},\qquad
E_{\rm CCD,k}=O_2 B D(\theta_k)A O_1E_{\rm SLM,k},
\]
\[
D_{pp}(\theta_k)=e^{i\phi_{\max}\theta_kG_p},\qquad
r_k=\operatorname{downsample}(|E_{\rm CCD,k}|^2).
\]

输入连续作用于整个 SLM：\(\Phi_p(u_k)=u_k(0.5+0.5M_p)\)，其中固定随机空间 mask \(M_p\in[0,1]\)，因此每个 SLM 像素都随 \(u_k\) 改变相位，而不是只切换部分阈值像素。空间 mask 的作用是避免整个 SLM 只获得相同全局相位；若 phase-only 输入在所有点严格相同，只形成公共相位，理想强度探测无法看到其变化。可选 `slm_encoding='binary_threshold'` 恢复以前的阈值编码。

\(O_1,O_2\) 是 objective Fourier pupil，\(A,B\) 为一次生成并固定的随机复振幅传播过程。局部高斯 \(G\) 的 support 占中间有效平面约 **1/10**；后续传播可使更大范围的 CCD 发生变化。\(D=1\) 于 support 之外。全平面 SLM 输入和局部热源是两个独立的空间概念。

若定义实际温升为 \(\Delta T_p=T_s\theta_kG_p\)，并假设热光系数和有效受热厚度在局部近似均匀，则
\[
\Delta n_p\approx\frac{dn}{dT}\Delta T_p,\qquad
\delta\phi_p\approx\frac{2\pi}{\lambda}L_{\rm eff}\frac{dn}{dT}T_s\theta_kG_p.
\]
模型用单个可调参数 \(\phi_{\max}\) 合并这些尚未标定的物理量；这给出从温度、折射率到相位的近似桥梁。

\(\theta_k\) 是内部 reservoir state；\(r_k\) 是相机观测向量，而非反馈状态。一个标量温度只提供一个独立热动态变量，256 个 CCD 特征不会自动变成 256 个独立的热记忆。热相位屏是温度到 transmission matrix 的有效近似，不是完整的热扩散或折射率求解。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from optical_physical_memory import PhysicalMemoryOptics, PhysicalNoise, RidgeReadout, nrmse

torch.set_num_threads(1)
plt.rcParams['figure.dpi'] = 100
noise = PhysicalNoise(pupil_rms_rad=0.18, slm_phase_std_rad=0.06,
                      photon_scale=5000, read_noise_electrons=3.0, shot_noise=True)
previous_noise = PhysicalNoise(pupil_rms_rad=0.06, slm_phase_std_rad=0.015,
                               photon_scale=50000, read_noise_electrons=1.0, shot_noise=True)
ideal = PhysicalNoise(pupil_rms_rad=0, slm_phase_std_rad=0,
                      photon_scale=50000, read_noise_electrons=0, shot_noise=False)
common = dict(optical_shape=(128, 128), slm_shape=(128, 128),
    input_shape=(32, 32), camera_shape=(16, 16), pupil_radius=(0.25, 0.25),
    heater_fraction=0.1, heater_phase_max=6.0, dt=1.0,
    thermal_gain=1.0, seed=42, noise_seed=123)
model = PhysicalMemoryOptics(**common, tau=8, beta=None, heat_mode='driven', noise=noise)
print(f'Heater support = {float(model.heater_support.float().mean()):.6f}')
print(f'a = {model.a:.6f}; beta = {model.beta:.6f}')
print('Input, SLM, optical, CCD features:', model.input_shape, model.slm_shape, model.optical_shape, model.camera_shape)

## 3. 检查 SLM、pupil 和 CCD

物镜的有限 pupil 是确定性滤波；额外的固定 pupil 波前误差、SLM 相位抖动、CCD Poisson shot noise 和读出噪声分别可调。与前一版相比，示例中的 pupil 像差从 0.06 增至 0.18 rad、SLM 抖动从 0.015 增至 0.06 rad、`photon_scale` 从 50,000 降至 5,000（提高相对散粒噪声）、读出噪声从 1 增至 3 电子。这些都是未经标定的示例；pupil 像差是静态器件差异，其余为逐次曝光噪声。每帧不做 min–max 归一化。

`|E_SLM|=1` 的白图是 phase-only SLM 的正常振幅；灰度相位图展示整个 SLM 的连续空间相位及抖动。

In [ ]:
sample_inputs = np.random.default_rng(7).random(50)
_, _, _, snap = model.run(sample_inputs, snapshot_at=49)
fig, axes = plt.subplots(2, 4, figsize=(14, 7), constrained_layout=True)
panels = [(snap['input_image'], 'continuous input phase pattern', 'gray', 0, 1),
          (snap['slm_phase'], 'full-SLM phase (rad)', 'gray', 0, np.pi),
          (np.abs(snap['slm_field']), '|exp(i phase)|', 'gray', 0, 1),
          (model.pupil1_support.cpu(), 'objective 1 pupil', 'gray', 0, 1),
          (model.heater_profile.cpu(), 'heater support ~10%', 'inferno', 0, 1),
          (snap['ccd_ideal'], 'ideal CCD', 'magma', None, None),
          (snap['ccd_measured'], 'noisy CCD', 'magma', None, None),
          (snap['features'], 'CCD feature r[k]', 'magma', None, None)]
for ax, (arr, title, cmap, vmin, vmax) in zip(axes.flat, panels):
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=9); ax.axis('off'); fig.colorbar(im, ax=ax, fraction=.045, pad=.03)
plt.show()
print('theta used for camera:', snap['theta_read'], 'next theta:', snap['theta_next'])
print('SLM amplitude min/max:', np.abs(snap['slm_field']).min(), np.abs(snap['slm_field']).max())
print('Mean expected CCD electrons per optical pixel:', snap['ccd_ideal'].mean()*noise.photon_scale)

# With jitter disabled, every SLM pixel has a nonzero input response.
check = PhysicalMemoryOptics(**common, tau=8, heat_mode='driven', noise=ideal)
_, _, _, low = check.run([0.25], snapshot_at=0)
_, _, _, high = check.run([0.75], snapshot_at=0)
coverage = np.mean(np.abs(high['slm_phase']-low['slm_phase']) > 1e-7)
print('Fraction of SLM pixels modulated by input change:', coverage)
assert coverage == 1.0

# Repeat a fixed input/temperature; this quantifies temporal noise remaining
# after 128×128 -> 16×16 pooling, not a calibrated experimental measurement.
for name, cfg in [('previous noise', previous_noise), ('strong noise', noise)]:
    repeat_model = PhysicalMemoryOptics(**common, tau=8, heat_mode='static', noise=cfg)
    repeats, _, _, _ = repeat_model.run(np.full(100, 0.5))
    relative_fluctuation = repeats.std(axis=0).mean()/repeats.mean()
    print(f'{name}: pooled-feature temporal std / mean = {relative_fluctuation:.2%}')

## 4. 脉冲写入与指数消退：验证公式时序

输入脉冲为 \(u_0=1\)，随后为零，初始 \(\theta_0=0\)。预期相机使用的温度依次为 \(\theta_0=0\)、\(\theta_1=\beta\)、\(\theta_2=a\beta\)，之后按 \(a\) 衰减。图中 0 值不是延迟错误，而是本实验规定曝光后再写入温度。

In [ ]:
thermal = PhysicalMemoryOptics(**common, tau=8, heat_mode='driven', noise=ideal)
pulse = np.r_[1.0, np.zeros(39)]
_, theta, theta_next, _ = thermal.run(pulse)
expected = np.r_[0.0, thermal.beta*thermal.a**np.arange(39)]
assert np.allclose(theta, expected)
assert np.allclose(theta_next, thermal.a*theta + thermal.beta*pulse)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.stem(np.arange(len(pulse)), theta, linefmt='C0-', markerfmt='C0o', basefmt=' ', label='theta[k]')
ax.plot(expected, 'C1--', label='analytic')
ax.set(xlabel='step k', ylabel='thermal state', title='Input pulse written into heat and fading')
ax.legend(); plt.show()
print('Maximum recursion error:', np.max(np.abs(theta-expected)))

## 5. 同一当前输入，不同输入历史

固定同一随机器件和 probe，在最后一步输入相同；比较之前 40 步全 0 与全 1 的热历史。为隔离记忆，这里关闭噪声，并与无热、静态热、即时响应对照。最后一步使用 \(\theta_k\) 而不是提前写入的 \(\theta_{k+1}\)。

In [ ]:
history0 = np.r_[np.zeros(40), 0.5]
history1 = np.r_[np.ones(40), 0.5]
history_results = {}
for mode in ['off', 'static', 'instant', 'driven']:
    m = PhysicalMemoryOptics(**common, tau=8, heat_mode=mode, noise=ideal)
    r0, t0, _, s0 = m.run(history0, snapshot_at=40)
    r1, t1, _, s1 = m.run(history1, snapshot_at=40)
    delta = np.linalg.norm(r1[-1]-r0[-1])/max(np.linalg.norm(r0[-1]), 1e-12)
    history_results[mode] = delta
    print(f'{mode:<10} theta=({t0[-1]:.4f}, {t1[-1]:.4f})  relative speckle feature difference={delta:.5f}')
assert history_results['off'] < 1e-7 and history_results['static'] < 1e-7
assert history_results['instant'] < 1e-7 and history_results['driven'] > 1e-5
delta_image = np.abs(s1['ccd_ideal']-s0['ccd_ideal'])
print('CCD fraction with |ΔI| > 5% of mean reference intensity:',
      np.mean(delta_image > .05*np.mean(s0['ccd_ideal'])))
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), constrained_layout=True)
for ax, arr, title in zip(axes, [s0['ccd_ideal'], s1['ccd_ideal'], delta_image],
                          ['zero heating history', 'one heating history', 'absolute CCD difference']):
    im=ax.imshow(arr, cmap='magma'); ax.set_title(title); ax.axis('off'); fig.colorbar(im, ax=ax, fraction=.045)
plt.show()

## 6. Mackey–Glass 单步预测

时间顺序划分：**前 2,000 步为训练段（含前 200 步 washout），随后 1,000 步为测试段**。训练段末尾 400 步只在训练内部用于选择 ridge 参数，选定后使用全部可用的 washout 后训练样本重新拟合。输入归一化只用内部拟合段确定；热状态跨训练和测试边界 **连续演化，不重置**。

readout 仅使用 \(r_k\) 与 bias，目标为 \(u_{k+1}\)。这里按**输入时刻**划分：测试输入为索引 2000–2999，其一步预测目标为索引 2001–3000（零起始索引）。为避免把测试区间的第一个数值当作训练标签，训练段的最后一个输入仍送入系统，但不用于拟合目标；因此 2,000 步训练段、200 步 washout 后，最终有 **1,799 个**监督训练样本。测试以真实输入驱动，属于 teacher-forced 预测。

对照包括无热、固定热、即时热、不同 \(\tau\) 的输入驱动热状态、噪声开关和 persistence。静态光学映射也可能预测自相关时间序列，因此请结合下节的独立随机输入延迟重建解释。

`previous noise` 对应前一版较小噪声，`strong noise` 对应本版较大噪声。两者的 pupil 静态像差也不同，因此整体差异不能全部归因于时间随机噪声。

每个条件的 \(W_{\rm out}\) 与 ESN 的线性读出一样用 ridge 回归求解：
\[
W_{\rm out}=(Z^TZ+\lambda P)^{-1}Z^TY,
\quad Z_k=[1,\widetilde r_k^T],\quad P=\operatorname{diag}(0,1,\ldots,1).
\]
特征均值和方差只由训练数据拟合，bias 不做正则化；代码通过求解线性方程而非显式求逆来计算权重。

In [ ]:
series = np.loadtxt(Path('MackeyGlass_t17.txt')).reshape(-1)
train_length, validation_length, test_length, washout = 2000, 400, 1000, 200
fit_end = train_length-validation_length
assert len(series) >= train_length+test_length+1
lo, hi = series[:fit_end].min(), series[:fit_end].max()
u_raw = (series[:train_length+test_length]-lo)/(hi-lo)
u = np.clip(u_raw, 0, 1)
y = (series[1:train_length+test_length+1]-lo)/(hi-lo)
train = np.arange(washout, fit_end-1)
valid = np.arange(fit_end, train_length-1)
fit_all = np.arange(washout, train_length-1)
test = np.arange(train_length, train_length+test_length)
print(f'Train steps: {train_length}, washout: {washout}, fit samples: {len(fit_all)}, test steps: {len(test)}')
print(f'Test input indices: {test[0]}–{test[-1]}; target indices: {test[0]+1}–{test[-1]+1}')
assert fit_all[-1]+1 < train_length and len(test)==test_length
print('Test input clipping fraction:', np.mean(u_raw[test] != u[test]))
print('Test correlation of u[k] and u[k+1]:', np.corrcoef(u[test], y[test])[0, 1])
cases = [
    ('No heat, ideal', 'off', 8, ideal),
    ('Thermal τ=8, ideal', 'driven', 8, ideal),
    ('Thermal τ=8, previous noise', 'driven', 8, previous_noise),
    ('No heat, strong noise', 'off', 8, noise),
    ('Static heat, strong noise', 'static', 8, noise),
    ('Instant heat, strong noise', 'instant', 8, noise),
    ('Thermal τ=2, strong noise', 'driven', 2, noise),
    ('Thermal τ=8, strong noise', 'driven', 8, noise),
    ('Thermal τ=32, strong noise', 'driven', 32, noise),
]
predictions, readouts = {}, {}
for label, mode, tau, noise_config in cases:
    m = PhysicalMemoryOptics(**common, tau=tau, heat_mode=mode, noise=noise_config)
    features, thermal_state, _, _ = m.run(u)
    lambdas = [0.1, 1.0, 10.0, 100.0]
    validation = [nrmse(y[valid], RidgeReadout(l).fit(features[train], y[train]).predict(features[valid])) for l in lambdas]
    alpha = lambdas[int(np.argmin(validation))]
    readout = RidgeReadout(alpha).fit(features[fit_all], y[fit_all])
    predictions[label], readouts[label] = readout.predict(features[test]), readout
    print(f'{label:<25} NRMSE={nrmse(y[test], predictions[label]):.4f}  ridge={alpha:g}')
print(f"{'Persistence':<25} NRMSE={nrmse(y[test], u[test]):.4f}")
print('W_out shape (bias + camera features):', readouts['Thermal τ=8, strong noise'].weights_.shape)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(15, 9), sharex=True, constrained_layout=True)
test_steps = np.arange(len(test))
for ax, label in zip(axes, ['No heat, strong noise', 'Static heat, strong noise', 'Thermal τ=2, strong noise', 'Thermal τ=8, strong noise']):
    ax.plot(test_steps, y[test], color='black', lw=1.0, label='target')
    ax.plot(test_steps, predictions[label], lw=0.9, label=label)
    ax.set_title(f'{label}: NRMSE={nrmse(y[test], predictions[label]):.4f}', fontsize=9)
    ax.legend(fontsize=8)
axes[-1].set_xlim(0, len(test))
axes[-1].set_xlabel('test time step'); plt.show()

weights = readouts['Thermal τ=8, strong noise'].weights_[1:].reshape(common['camera_shape'])
fig, ax = plt.subplots(figsize=(5, 4)); im=ax.imshow(weights, cmap='coolwarm')
ax.set_title('W_out on standardized speckle features'); fig.colorbar(im, ax=ax); plt.show()

## 7. 独立随机输入的历史重建

产生 3,000 个 i.i.d. 输入，仅根据当前 \(r_k\) 重建 \(u_{k-d}\)：前 2,000 步为训练段（前 200 步 washout），后 1,000 步测试。这比 Mackey–Glass 一步预测更直接检验过去输入能否从 speckle 读出。负的测试 \(R^2\) 表示比测试均值基线更差。不同 \(\tau\) 保持热稳态增益相同。实验包含相机噪声，应进一步更换多个随机种子和参数确认稳健性。

In [ ]:
random_u = np.random.default_rng(2026).random(3000)
memory_cases = cases[3:]
delays = np.array([1, 2, 4, 8, 16]); split_memory = 2000
memory_scores = {}
for label, mode, tau, noise_config in memory_cases:
    m = PhysicalMemoryOptics(**common, tau=tau, heat_mode=mode, noise=noise_config)
    features, _, _, _ = m.run(random_u)
    values = []
    for delay in delays:
        tr = np.arange(200, split_memory); te = np.arange(split_memory, len(random_u))
        pred = RidgeReadout(10.0).fit(features[tr], random_u[tr-delay]).predict(features[te])
        target = random_u[te-delay]
        values.append(1-np.sum((target-pred)**2)/np.sum((target-target.mean())**2))
    memory_scores[label] = np.array(values)
    print(f'{label:<25}', np.round(values, 3))
fig, ax = plt.subplots(figsize=(9, 4))
for label, result in memory_scores.items(): ax.plot(delays, result, 'o-', label=label)
ax.axhline(0, color='0.5', lw=.8); ax.set(xlabel='delay d', ylabel='held-out R²')
ax.legend(fontsize=7); ax.grid(alpha=.2); plt.show()

## 8. 非线性是否可读出？

使用独立随机输入 \(x_k=2u_k-1\)，测试二阶 \(P_2(x)=(3x^2-1)/2\) 与当前/过去乘积。前 2,000 步为训练段（含 200 步 washout），后 1,000 步测试；训练段内部划分验证集。分别比较无热、不同 \(\tau\) 的热记忆，以及固定 probe、仅加热序列变化的条件。固定 probe 帮助区分 SLM 当前输入编码和“热相位→干涉→CCD”的非线性。每个任务只用验证段选择 ridge，报告未参与拟合的测试 \(R^2\)。

In [ ]:
nl_u = np.random.default_rng(213).random(3000)
x = 2*nl_u-1
targets = {'current P2': .5*(3*x*x-1),
           'delay-1 P2': .5*(3*np.roll(x, 1)**2-1),
           'current × delay-1': x*np.roll(x, 1)}
nl_train = np.arange(200, 1600); nl_valid = np.arange(1600, 2000); nl_test = np.arange(2000, len(nl_u))
nl_cases = [('No heat', 'off', 2, False), ('Thermal τ=2', 'driven', 2, False),
            ('Thermal τ=8', 'driven', 8, False), ('τ=2 fixed probe', 'driven', 2, True)]
def score_r2(target, prediction, indices):
    y0 = target[indices]
    return 1-np.sum((y0-prediction)**2)/np.sum((y0-y0.mean())**2)
print(f"{'Condition':<22}" + ''.join(f'{name:>21}' for name in targets))
for label, mode, tau, fixed_probe in nl_cases:
    m = PhysicalMemoryOptics(**common, tau=tau, heat_mode=mode, noise=noise)
    probe = np.full(len(nl_u), 0.5) if fixed_probe else nl_u
    features, _, _, _ = m.run(probe, heat_inputs=nl_u)
    result = []
    for target in targets.values():
        candidates = [1.0, 10.0, 100.0, 1000.0]
        validation = [score_r2(target, RidgeReadout(a).fit(features[nl_train], target[nl_train]).predict(features[nl_valid]), nl_valid) for a in candidates]
        best = candidates[int(np.argmax(validation))]
        fit = np.r_[nl_train, nl_valid]
        prediction = RidgeReadout(best).fit(features[fit], target[fit]).predict(features[nl_test])
        result.append(score_r2(target, prediction, nl_test))
    print(f'{label:<22}' + ''.join(f'{v:>21.3f}' for v in result))

## 9. 结果的边界

1. 这个版本的递推变量只有 \(\theta\)；没有 previous speckle、电子 leak 或输出到 SLM 的反馈。
2. 固定随机传播中插入局部相位屏，是 \(\Delta T\rightarrow\Delta n\rightarrow\Delta\phi\rightarrow T\) 的简化有效模型；\(\phi_{\max}\) 和输入热耦合尚未按材料、波长和厚度标定。
3. 假设单一空间高斯形状和单一弛豫时间；扩展为多个空间区域、时间常数或热扩散方程需要另行建模。
4. 图与指标只展示一组器件和随机种子，不能单凭一次仿真推断实验性能。